# Part 1: Data Preprocessing

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the dataset
file_path = '/Users/luischan/Downloads/survey.csv'
df = pd.read_csv(file_path)

# Display the first few rows to inspect the dataset
print(df.head())


             Timestamp  Age  Gender         Country state self_employed  \
0  2014-08-27 11:29:31   37  Female   United States    IL           NaN   
1  2014-08-27 11:29:37   44       M   United States    IN           NaN   
2  2014-08-27 11:29:44   32    Male          Canada   NaN           NaN   
3  2014-08-27 11:29:46   31    Male  United Kingdom   NaN           NaN   
4  2014-08-27 11:30:22   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often            6-25  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely            6-25  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

                leave mental_health_consequence phys_health_consequence  \
0       Somewhat easy                        No                      No   
1 

In [2]:
# Standardize column names: lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Display standardized column names
print(df.columns)

Index(['timestamp', 'age', 'gender', 'country', 'state', 'self_employed',
       'family_history', 'treatment', 'work_interfere', 'no_employees',
       'remote_work', 'tech_company', 'benefits', 'care_options',
       'wellness_program', 'seek_help', 'anonymity', 'leave',
       'mental_health_consequence', 'phys_health_consequence', 'coworkers',
       'supervisor', 'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence', 'comments'],
      dtype='object')


In [3]:
# Check for missing values in each column
print(df.isnull().sum())


timestamp                       0
age                             0
gender                          0
country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1095
dtype: int64


In [4]:
# Fill missing values in categorical columns with the most frequent category
categorical_columns = df.select_dtypes(include=['object']).columns

for col in categorical_columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Fill missing values in numerical columns with the median
numerical_columns = df.select_dtypes(include=['number']).columns

for col in numerical_columns:
    df[col].fillna(df[col].median(), inplace=True)

# Verify that there are no more missing values
print(df.isnull().sum())


timestamp                    0
age                          0
gender                       0
country                      0
state                        0
self_employed                0
family_history               0
treatment                    0
work_interfere               0
no_employees                 0
remote_work                  0
tech_company                 0
benefits                     0
care_options                 0
wellness_program             0
seek_help                    0
anonymity                    0
leave                        0
mental_health_consequence    0
phys_health_consequence      0
coworkers                    0
supervisor                   0
mental_health_interview      0
phys_health_interview        0
mental_vs_physical           0
obs_consequence              0
comments                     0
dtype: int64


/var/folders/p6/dvcp9zk51c534gxpxqw62v680000gp/T/ipykernel_38626/3453467860.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/var/folders/p6/dvcp9zk51c534gxpxqw62v680000gp/T/ipykernel_38626/3453467860.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa

In [5]:
# Inspect unique values in the 'gender' column
print(df['gender'].unique())

# Standardize 'gender' entries
def clean_gender(value):
    if value.lower() in ['male', 'm']:
        return 'Male'
    elif value.lower() in ['female', 'f']:
        return 'Female'
    else:
        return 'Other'

df['gender'] = df['gender'].apply(clean_gender)

# Verify cleaned gender values
print(df['gender'].unique())


['Female' 'M' 'Male' 'male' 'female' 'm' 'Male-ish' 'maile' 'Trans-female'
 'Cis Female' 'F' 'something kinda male?' 'Cis Male' 'Woman' 'f' 'Mal'
 'Male (CIS)' 'queer/she/they' 'non-binary' 'Femake' 'woman' 'Make' 'Nah'
 'All' 'Enby' 'fluid' 'Genderqueer' 'Female ' 'Androgyne' 'Agender'
 'cis-female/femme' 'Guy (-ish) ^_^' 'male leaning androgynous' 'Male '
 'Man' 'Trans woman' 'msle' 'Neuter' 'Female (trans)' 'queer'
 'Female (cis)' 'Mail' 'cis male' 'A little about you' 'Malr' 'p' 'femail'
 'Cis Man' 'ostensibly male, unsure what that really means']
['Female' 'Male' 'Other']


In [6]:
# Create a 'stigma_score' by aggregating responses to stigma-related questions
stigma_columns = ['mental_health_consequence', 'phys_health_consequence', 'coworkers', 'supervisor']

# Convert responses to numerical values
stigma_mapping = {
    'No': 0,
    'Maybe': 1,
    'Yes': 2
}

for col in stigma_columns:
    df[col] = df[col].map(stigma_mapping)

# Calculate the stigma score as the sum of the mapped values
df['stigma_score'] = df[stigma_columns].sum(axis=1)

# Display the new stigma score column
print(df[['stigma_score']].head())


   stigma_score
0           2.0
1           1.0
2           4.0
3           4.0
4           2.0


In [7]:
# Create an 'employer_support_score' by aggregating support-related columns
support_columns = ['benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity']

# Convert responses to numerical values
support_mapping = {
    'No': 0,
    'Yes': 2,
    'Don\'t know': 1
}

for col in support_columns:
    df[col] = df[col].map(support_mapping)

# Calculate the employer support score as the sum of the mapped values
df['employer_support_score'] = df[support_columns].sum(axis=1)

# Display the new employer support score column
print(df[['employer_support_score']].head())


   employer_support_score
0                     6.0
1                     4.0
2                     1.0
3                     2.0
4                     5.0


In [9]:
# Save the preprocessed dataset to a new CSV file
preprocessed_file_path = '/Users/luischan/Downloads/preprocessed_survey.csv'
df.to_csv(preprocessed_file_path, index=False)

print(f"Preprocessed dataset saved to {preprocessed_file_path}")

Preprocessed dataset saved to /Users/luischan/Downloads/preprocessed_survey.csv


# **Conclusion: Data Preprocessing Steps and Results**

---

## **Overview**

This preprocessing step involved standardizing, cleaning, and enhancing the *Workplace Mental Health in Tech* dataset. The goal was to prepare the dataset for meaningful analysis by addressing missing data, cleaning inconsistent entries, and engineering new features.

---

## **Steps Taken**

### **1. Standardization**
- **Column Names**: Converted all column names to lowercase and replaced spaces with underscores for consistency.

### **2. Data Cleaning**
- **Missing Values**:
  - Filled **categorical columns** with the most frequent category.
  - Filled **numerical columns** with the median value.
- **Gender Standardization**:
  - Cleaned and consolidated various gender entries into three categories: `Male`, `Female`, and `Other`.

### **3. Feature Engineering**
- **Stigma Score**:
  - Created by aggregating responses from stigma-related questions (`mental_health_consequence`, `phys_health_consequence`, `coworkers`, `supervisor`).
  - Values were mapped as follows:  
    - `No` → 0, `Maybe` → 1, `Yes` → 2.
- **Employer Support Score**:
  - Created by aggregating responses related to employer support (`benefits`, `care_options`, `wellness_program`, `seek_help`, `anonymity`).
  - Values were mapped as follows:  
    - `No` → 0, `Don’t know` → 1, `Yes` → 2.

### **4. Saving the Data**
- Exported the preprocessed dataset to `preprocessed_survey.csv`.

---

## **Results**

### **1. Missing Data**
- All missing values were successfully handled. There are no missing entries left in the dataset.

### **2. Gender Cleaning**
- The `gender` column was cleaned and standardized to three categories:  
  - `['Female', 'Male', 'Other']`

### **3. Feature Engineering**
- **Stigma Score**:  
  Represents the overall stigma associated with mental health in the workplace.  
  - Example scores:  
    ```
    2.0, 1.0, 4.0, 4.0, 2.0
    ```
- **Employer Support Score**:  
  Represents the level of mental health support provided by employers.  
  - Example scores:  
    ```
    6.0, 4.0, 1.0, 2.0, 5.0
    ```

---

## **Key Findings**

1. **Data Consistency**:
   - Missing values and inconsistent entries were effectively resolved, ensuring a clean dataset ready for analysis.

2. **Stigma and Employer Support**:
   - Derived scores provide quantitative measures for analyzing the relationship between stigma, employer support, and mental health outcomes.

3. **Gender Diversity**:
   - The dataset contained a wide variety of gender responses, which were consolidated to simplify analysis while retaining inclusivity.

---

## **Next Steps**

### **1. Data Analysis**
- Perform detailed analyses, including:
  - Correlation analysis between stigma and employer support.
  - Logistic regression to predict treatment-seeking behavior.
  - Sentiment analysis on open-ended comments.

### **2. Web Interface Development**
- Create a Flask API to serve analysis results.
- Develop a web interface to allow users to select analyses, input parameters, and visualize results.

### **3. Visualization**
- Generate graphs such as heatmaps, ROC curves, and choropleth maps to illustrate findings.

### **4. Report and Presentation**
- Document findings and present the analysis through a clear, concise 15-minute presentation.
